# GreenTest: JS

This is the **JS** GreenTest notebook. Every language in this repo follows the
same pattern: bootstrap the language, generate a small static site, serve
it locally, and verify it's actually being served correctly.

This one verifies [vanilla-compost](https://github.com/EcologyComputing/vanilla-compost) using Node.js, and walks you through the step-by-step bash commands and concepts.

Steps:
1. Point this notebook at your vanilla-compost clone
2. Confirm it's actually cloned
3. Leave a note about this run
4. Copy `greenTest-Message.md` to vanilla-compost posts folder
5. Generate `posts.html` from the markdown posts using Node.js
6. Serve the site locally using Node.js
7. Verify the generated page is actually being served correctly
8. Clean up

## 1. Point this notebook at your vanilla-compost clone

GreenTest assumes you cloned vanilla-compost as a sibling of this repo, so
the same folder that has `greenTest/` should also have `vanilla-compost/`.
This cell runs in **bash**. If your clone of vanilla compost lives somewhere else, change the path in the bash script below.

In [ ]:
%%bash 
export VANILLA_COMPOST="../../vanilla-compost"
echo "Testing vanilla-compost at: $VANILLA_COMPOST"

## 2. Confirm the repo is cloned

This next script is also in bash. It checks that expected files exist and that the folder is a real `git clone`.

In [ ]:
%%bash
VANILLA_COMPOST="../../vanilla-compost"
if [ -f "$VANILLA_COMPOST/README.md" ] && [ -f "$VANILLA_COMPOST/src/generate_posts.js" ]; then
    echo "Repo layout looks right (found README.md and src/generate_posts.js)."
else
    echo "That path doesn't look like a vanilla-compost clone with generate_posts.js: $VANILLA_COMPOST"
    exit 1
fi

if git -C "$VANILLA_COMPOST" rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    echo "Confirmed: this is a real git clone, not just a folder of files."
    git -C "$VANILLA_COMPOST" remote get-url origin 2>/dev/null && echo "Its 'origin' remote points there." || echo "No 'origin' remote set."
else
    echo "Warning: no .git found there."
fi

## 3. Leave a note for this run

Edit the text `"""` marks below with anything about this run. It gets appended to `greenTest-Message.md`.

In [ ]:
notes = """
Write your notes, or just "Hello, World!" here before running the rest of the notebook.
"""

In [ ]:
from datetime import datetime

log_path = "greenTest-Message.md"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"## {timestamp}\n\n{notes.strip()}\n\n")

print(f"Notes appended to {log_path}")

## 4. Copy `greenTest-Message.md` to vanilla-compost posts folder

We copy our run log into the blog's `posts/` folder so it will be recognized as a post by the generator script.

In [ ]:
import os
import shutil

VANILLA_COMPOST = os.environ.get("VANILLA_COMPOST", "../../vanilla-compost")
dest_dir = os.path.join(VANILLA_COMPOST, "src", "posts")
os.makedirs(dest_dir, exist_ok=True)
shutil.copy("greenTest-Message.md", os.path.join(dest_dir, "greenTest-Message.md"))
print(f"Copied greenTest-Message.md to {dest_dir}/")

## 5. Generate `posts.html`

This cell runs vanilla-compost's `generate_posts.js` using Node.js.

In [ ]:
import subprocess

result = subprocess.run(
    ["node", os.path.join(VANILLA_COMPOST, "src", "generate_posts.js")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_posts.js failed - see output above.")

Peek at what it generated:

In [ ]:
with open(os.path.join(VANILLA_COMPOST, "src", "posts.html"), encoding="utf-8") as f:
    generated = f.read()

start = generated.find('<p class="lead">')
end = generated.find('</p>', start) + len('</p>')
print(generated[start:end] if start != -1 else generated)

## 6. Serve the site locally

This starts our static file server (`node server.js`) in the background.

In [ ]:
import time

server = subprocess.Popen(
    ["node", "server.js", os.path.join(VANILLA_COMPOST, "src")],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)  # give it a moment to start
print(f"Server started (pid {server.pid}) at http://localhost:8080/")

## 7. Verify it's serving the update correctly

We fetch the page from the Node.js server and verify it matches the generated HTML.

In [ ]:
import urllib.request

with urllib.request.urlopen("http://localhost:8080/posts.html") as response:
    served = response.read().decode("utf-8")

assert served == generated, "Served posts.html doesn't match what generate_posts.js just wrote."
assert "hello-compost" in served, "Expected the hello-compost post to be listed."

print("Green: the Node.js server is serving the freshly generated posts.html.")

## 8. Clean up

In [ ]:
server.terminate()
server.wait()
print("Server stopped.")